In [1]:
import numpy as np
import mitsuba as mi

mi.set_variant("scalar_acoustic")

# ---------------------------------------------------------------------------
# Channel naming helper
# ---------------------------------------------------------------------------
# This MUST stay byte-identical to:
#   - directivity_csv_to_misuka_exr.ipynb's frequency_to_channel_name()
#   - test_speaker_directivity.py's frequency_to_channel_name()
#   - speaker.cpp's frequency_to_channel_name()
# Any divergence will cause channel-not-found errors at scene load.
#
# Note: the trailing '.' is the OpenEXR layer/channel separator. The part
# before the dot (e.g. '2000_Hz') is the 'layer' and the part after is the
# 'channel-within-layer' (empty here). Bitmap.split() in speaker.cpp uses
# this structure to return one single-channel bitmap per frequency.
def frequency_to_channel_name(freq):
    """Convert a frequency in Hz to its canonical EXR channel name (e.g. 2000.0 -> '2000_Hz.')."""
    return f"{freq:g}_Hz."

# ---------------------------------------------------------------------------
# Build a synthetic directivity texture: front ring (theta in [0, 10]) = 1.0
# ---------------------------------------------------------------------------
TEST_FREQUENCY = 2000
FILENAME       = "textures/test_2000.exr"

data = np.full((181, 360), 0.0, dtype=np.float32)
data[0:10, :] = 1.0  # entire front ring = theta in [0, 10] degrees

# Add singleton channel axis: (181, 360) -> (181, 360, 1)
data_3d = data[:, :, None]

# Write as a single-channel multi-channel EXR with the canonical channel name.
# Explicit MultiChannel pixel format prevents Mitsuba from auto-classifying as
# luminance (Y) and overriding our channel name.
import os
os.makedirs(os.path.dirname(FILENAME), exist_ok=True)

bmp = mi.Bitmap(
    data_3d,
    pixel_format=mi.Bitmap.PixelFormat.MultiChannel,
    channel_names=[frequency_to_channel_name(TEST_FREQUENCY)],
)
bmp.write(FILENAME)

In [2]:
import numpy as np
import drjit as dr
import mitsuba as mi

mi.set_variant("scalar_acoustic")
mi.set_log_level(mi.LogLevel.Warn)

T = mi.ScalarTransform4f

MAX_TIME      = 0.03
SAMPLING_RATE = 10000
N_TIME_BINS   = int(MAX_TIME * SAMPLING_RATE)

integrator = mi.load_dict({
    'type': 'acoustic_path',
    'max_depth': 1,
    'max_time': MAX_TIME,
    'speed_of_sound': 343,
})

spp = 2**5

# Path to the multi-channel EXR written in the previous cell
DIRECTIVITY_FILE = "textures/test_2000.exr"
TEST_FREQUENCY   = 2000


def make_scene(speaker_pos, mic_pos, target):
    scene = mi.load_dict({
        'type': 'scene',

        'microphone': {
            'type': 'microphone',
            'to_world': T().translate(mic_pos),
            'film': {
                'type': 'tape',
                'frequencies': str(TEST_FREQUENCY),
                'time_bins': N_TIME_BINS,
                'rfilter': {'type': 'box'},
            },
            'sampler': {'type': 'stratified'},
        },

        'speaker': {
            'type': 'sphere',
            'to_world': T().look_at(origin=speaker_pos, target=target, up=[0, 0, 1]).scale(1.0),
            'emitter': {
                'type': 'speaker',
                'radiance': 1.0,
                'directivity_file': DIRECTIVITY_FILE,
                'frequencies': str(TEST_FREQUENCY),
            },
        },
    })
    print(mi.traverse(scene)['speaker.to_world'])
    return mi.render(scene, integrator=integrator, seed=0, spp=spp) / spp


SPEAKER_POS  = mi.ScalarVector3f(0, 0, 0)
MIC_POS      = mi.ScalarVector3f(3, 0, 0)
ABSORBER_POS = mi.ScalarVector3f(3, 0, 3)


'''
etc_facing = make_scene(speaker_pos=SPEAKER_POS,
                        mic_pos=MIC_POS,
                        target=MIC_POS)
                        '''

etc_rotated = make_scene(speaker_pos=SPEAKER_POS,
                         mic_pos=MIC_POS,
                         target=ABSORBER_POS)


# energy_facing = float(np.array(etc_facing).sum())
# energy_rotated = float(np.array(etc_rotated).sum())

# energy_facing, energy_rotated

# Rohdaten aus der Matrix (ohne Skalierung)
origin   = np.array([0, 0, 0], dtype=float)
target   = np.array([3, 0, 3], dtype=float)
up_hint  = np.array([0, 0, 1], dtype=float)

# Schritt 1: forward
forward = target - origin
forward = forward / np.linalg.norm(forward)
print(f"forward = {forward}")

# Schritt 2: right
right = np.cross(up_hint, forward)
right = right / np.linalg.norm(right)
print(f"right   = {right}")

# Schritt 3: up korrigiert
up_corrected = np.cross(forward, right)
print(f"up      = {up_corrected}")

print()
print("Erwartete Matrix (Spalten = right, up, forward):")
print(f"  Spalte 0 (right):   {right}")
print(f"  Spalte 1 (up):      {up_corrected}")
print(f"  Spalte 2 (forward): {forward}")

RuntimeError: ​[BitmapTexture] The texture needs to have a known pixel format (Y[A], RGB[A], XYZ[A] are supported).